# InflationShip — end-to-end notebook
**Goal:** Build a pipeline that collects Port of Los Angeles & Long Beach TEU monthly data (2015–2025), fetches CPI/PPI/Import Price/GSCPI/BDI where available, cleans & aligns monthly series, seasonally adjusts TEUs, computes YoY growth, tests lead–lag relationships, runs Granger causality, fits VAR, and runs category-level regressions.

**Notes**
- PDF parsing is heuristic and may require minor adjustments for some yearly PDFs.
- `tabula-py` requires Java; `pdfplumber` is a lighter dependency for text extraction.
- If scraping fails, download POLA and POLB yearly tables manually and save them to the `inflationship_output/` folder as `pola_manual.csv` / `polb_manual.csv`.


### Imports and user settings
This cell imports libraries and sets start/end years, output folder, and the CPI series mapping.


In [51]:
import os, time, re, warnings
from datetime import datetime
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

from pandas_datareader import data as pdr
import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.api import VAR

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid', context='talk')

warnings.simplefilter(action='ignore', category=FutureWarning)

# -------------------------
# USER SETTINGS (edit these)
# -------------------------
OUTDIR = "inflationship_output"
os.makedirs(OUTDIR, exist_ok=True)

START_YEAR = 2015
END_YEAR = 2025
START_DATE = f"{START_YEAR}-01-01"
END_DATE = f"{END_YEAR}-12-31"

# CPI categories (FRED codes). Set None to skip a specific series.
CPI_CATEGORIES = {
    "CPI_All": "CPIAUCSL",       # All items, seasonally adjusted
    "CPI_Food": "CPIFABSL",
    "CPI_Apparel": "CPIAPPSL",
    "CPI_Vehicles":"CUSR0000SETA01",      # set a FRED series if you want vehicle CPI
    "CPI_Pharma": "CPIMEDSL",
    "CPI_Household": "CPIHOSSL"
}

IMPORT_PRICE_CODE = "IR"  # Import Price Index (headline)
GSCPI_XLSX_URL = "https://www.newyorkfed.org/medialibrary/research/interactives/gscpi/downloads/gscpi_data.xlsx"
bdifname = "BDI_Data.csv"              # exact file name in your OUTDIR
BDI_SOURCE = os.path.join(OUTDIR, bdifname).strip()


### Helper functions
- `fetch_fred_series()` fetches a series from FRED.
- `seasonal_adjust_series()` tries X-13 (if installed) then falls back to STL.
- `compute_yoy()` computes YoY percent change.


In [52]:
def fetch_fred_series(symbol, start=START_DATE, end=END_DATE):
    """Fetch a single FRED series and return a DataFrame (symbol column)."""
    try:
        s = pdr.DataReader(symbol, "fred", start, end)
        s.columns = [symbol]
        return s
    except Exception as e:
        print(f"[FRED] Failed to fetch {symbol}: {e}")
        return None

def seasonal_adjust_series(series, method='x13'):
    """Seasonally adjust a monthly pd.Series. Returns seasonally adjusted series aligned to original index."""
    try:
        if method == 'x13':
            res = sm.tsa.x13_arima_analysis(series.dropna())
            return res.seasadj.reindex(series.index)
    except Exception as e:
        # X13 may not be installed in environment
        # print("X13 unavailable:", e)
        pass
    # STL fallback
    try:
        stl = sm.tsa.STL(series.dropna(), period=12, robust=True)
        res = stl.fit()
        seasadj = (series.dropna() - res.seasonal).reindex(series.index)
        return seasadj
    except Exception as e:
        # Last resort: simple 12-month centered moving average detrend
        ma = series.rolling(window=12, center=True).mean()
        return (series - (ma - series.mean())).fillna(method='bfill').fillna(method='ffill')

def compute_yoy(series):
    """Compute YoY percent change (12-month)."""
    return series.pct_change(12) * 100


### Fetch CPI categories, Import Price Index (IR), and NYFed GSCPI
- The code pulls the CPI categories defined earlier from FRED.
- GSCPI is fetched from the NY Fed downloadable spreadsheet.
- If you have a BDI CSV, you can point `BDI_SOURCE` to it.


In [53]:
# --- Fetch all series into a dict ---
fred_series = {}
# CPI categories
for name, code in CPI_CATEGORIES.items():
    if code:
        s = fetch_fred_series(code)
        if s is not None:
            fred_series[name] = s

# Import price
imp = fetch_fred_series(IMPORT_PRICE_CODE)
if imp is not None:
    fred_series['ImportPrice'] = imp

# GSCPI
gscpi_df = fetch_gscpi()
if gscpi_df is not None:
    fred_series['GSCPI'] = gscpi_df

# BDI (fetched but might not use as already capturing the local cost changes in GSCPI)
bdi_df = load_bdi_source(BDI_SOURCE)
if bdi_df is not None:
    fred_series['BDI'] = bdi_df

if not fred_series:
    raise RuntimeError("No series fetched. Check connectivity and your config variables.")

# --- Clean & safe-concat ---
cleaned = []
issues = {}
for key, df in fred_series.items():
    if df is None:
        continue
    if isinstance(df, pd.Series):
        df = df.to_frame()
    # ensure datetime index
    try:
        df.index = pd.to_datetime(df.index)
    except Exception as e:
        # try to use a 'date' column if present
        try:
            date_col = next((c for c in df.columns if 'date' in c.lower()), None)
            if date_col is not None:
                df = df.rename(columns={date_col:'date'}).set_index(pd.to_datetime(df['date'], errors='coerce'))
            else:
                df = df.reset_index()
                df.index = pd.to_datetime(df.iloc[:,0])
        except Exception as e2:
            issues[key] = f"index->datetime failed: {e2}"
    # align to monthly period start
    try:
        df.index = df.index.to_period('M').to_timestamp()
    except Exception:
        pass
    # drop exact duplicate index rows (keep first)
    if df.index.duplicated().any():
        issues[key] = issues.get(key, "") + " duplicates dropped"
        df = df[~df.index.duplicated(keep='first')]
    # pick numeric column (or first column)
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        num_cols = [df.columns[0]]
    clean = df[[num_cols[0]]].copy()
    clean.columns = [key]
    cleaned.append(clean)

if not cleaned:
    raise RuntimeError("No cleaned series available to concat after processing.")

fred_df = pd.concat(cleaned, axis=1)
# save minimal output
fred_df = fred_df.loc[(fred_df.index >= pd.to_datetime(START_DATE)) & (fred_df.index <= pd.to_datetime(END_DATE))]
if fred_df.empty:
    raise RuntimeError(f"No rows remain after trimming to {START_DATE} - {END_DATE}.")

outpath = os.path.join(OUTDIR, "fred_and_gscpi_raw_fixed.csv")
fred_df.to_csv(outpath)
print(f"Created {outpath} with shape {fred_df.shape}")
if issues:
    print("Issues noted for some series (see keys):")
    for k,v in issues.items():
        print(f" - {k}: {v}")

Created inflationship_output/fred_and_gscpi_raw_fixed.csv with shape (130, 9)
Issues noted for some series (see keys):
 - GSCPI:  duplicates dropped
